# Pipeline de Alfabetização no Brasil — Notebook 3: Gold Layer

**Tech Challenge Fase 2 — FIAP POSTECH**

---

## Objetivo

Construção da camada **Gold**: datasets analíticos prontos para:
- Dashboards executivos
- Consultas SQL via Athena
- Treinamento de modelos de Machine Learning

## Datasets Gold Produzidos

| Dataset | Descrição |
|---|---|
| `gold_indicador_municipio` | Indicador atual + metas + gap por município |
| `gold_evolucao_temporal_uf` | Série histórica por UF |
| `gold_ranking_uf` | Ranking de UFs no ano mais recente |
| `gold_municipios_risco` | Municípios mais distantes da meta 2030 |
| `gold_comparativo_meta_brasil` | Evolução nacional vs meta 2030 |

## 1. Imports e Configuração

In [ ]:
import os
import io
import logging
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import boto3

try:
    from dotenv import load_dotenv
    _env_candidates = [
        Path.home() / "Desktop/tech-challenge-fase2/.env",
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
    ]
    for _p in _env_candidates:
        if _p.exists():
            load_dotenv(_p)
            break
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

S3_BUCKET    = os.getenv("S3_BUCKET_NAME",       "tech-challenge-alfabetizacao-01")
AWS_REGION   = os.getenv("AWS_DEFAULT_REGION",   "us-east-1")
USE_AWS      = os.getenv("USE_AWS", "false").lower() == "true"
LOCAL_SILVER = Path.home() / "Desktop/tech-challenge-fase2/data/silver"
LOCAL_GOLD   = Path.home() / "Desktop/tech-challenge-fase2/data/gold"
LOCAL_GOLD.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
print(f"Modo : {'AWS S3' if USE_AWS else 'LOCAL'}")
print(f"Bucket: {S3_BUCKET}")
print(f"USE_AWS env: {os.getenv('USE_AWS')}")

## 2. Leitura da Camada Silver

In [ ]:
def read_silver_local(name: str) -> pd.DataFrame:
    """Lê todos os Parquets de uma tabela Silver (incluindo partições)."""
    base = LOCAL_SILVER / name
    if not base.exists():
        logger.error("Silver não encontrado: %s. Execute notebook 02 primeiro.", base)
        return pd.DataFrame()

    frames = []
    for path in base.rglob("*.parquet"):
        df = pd.read_parquet(path)
        # Recupera partições do path (Hive-style)
        for part in path.parts:
            if "=" in part:
                col, val = part.split("=", 1)
                df[col] = val
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    result = pd.concat(frames, ignore_index=True)
    # Remove metadados de processamento
    result = result.drop(columns=["_data_processamento"], errors="ignore")
    return result


silver_mun = read_silver_local("alfabetizacao_municipio")
silver_uf  = read_silver_local("alfabetizacao_uf")
meta_brasil = read_silver_local("meta_brasil")

# Converte tipos após remontar partições
for df in [silver_mun, silver_uf]:
    if "ano" in df.columns:
        df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
    for col in ["taxa_alfabetizacao", "gap_meta_uf_2030", "gap_meta_municipio_2030",
                "media_portugues", "meta_uf_2030", "meta_mun_2030"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Silver município: {len(silver_mun):,} linhas")
print(f"Silver UF       : {len(silver_uf):,} linhas")
print(f"Meta Brasil     : {len(meta_brasil):,} linhas")

## 3. Construção dos Datasets Gold

### 3.1 Gold: Indicador por Município (dataset principal)

In [ ]:
def build_gold_municipio(df: pd.DataFrame) -> pd.DataFrame:
    """
    Dataset analítico por município:
    - Taxa de alfabetização atual
    - Metas 2024-2030
    - Gaps em relação às metas
    - Categoria de risco
    """
    cols = [c for c in [
        "id_municipio", "sigla_uf", "ano", "serie", "rede",
        "taxa_alfabetizacao", "media_portugues",
        "meta_mun_2030", "meta_uf_2030", "meta_brasil_2030",
        "gap_meta_municipio_2030", "gap_meta_uf_2030", "atingiu_meta_uf",
        "proporcao_aluno_nivel_0", "proporcao_aluno_nivel_1",
        "proporcao_aluno_nivel_2", "proporcao_aluno_nivel_3"
    ] if c in df.columns]

    gold = df[cols].copy()

    # Categoria de risco para IA
    if "gap_meta_uf_2030" in gold.columns:
        gold["categoria_risco"] = pd.cut(
            gold["gap_meta_uf_2030"],
            bins=[-float("inf"), -20, -10, 0, float("inf")],
            labels=["critico", "alto", "moderado", "meta_atingida"]
        ).astype(str)

    return gold


gold_municipio = build_gold_municipio(silver_mun)
print(f"Gold município: {len(gold_municipio):,} linhas")
display(gold_municipio.head(5))

### 3.2 Gold: Evolução Temporal por UF

In [ ]:
def build_gold_evolucao_uf(df: pd.DataFrame) -> pd.DataFrame:
    """
    Série histórica por UF: média da taxa e contagem de municípios.
    Base para análise de desigualdade regional.
    """
    if not {"ano", "sigla_uf", "taxa_alfabetizacao"}.issubset(df.columns):
        return pd.DataFrame()

    agg = (
        df.groupby(["ano", "sigla_uf"], dropna=False)
        .agg(
            media_taxa_alfabetizacao   = ("taxa_alfabetizacao", "mean"),
            mediana_taxa_alfabetizacao = ("taxa_alfabetizacao", "median"),
            desvio_padrao              = ("taxa_alfabetizacao", "std"),
            total_municipios           = ("id_municipio",       "nunique"),
        )
        .reset_index()
        .round(2)
    )

    # Adiciona meta UF 2030 (referência)
    if "meta_uf_2030" in df.columns:
        meta_ref = (
            df.groupby(["ano", "sigla_uf"])["meta_uf_2030"]
            .first().reset_index()
        )
        agg = agg.merge(meta_ref, on=["ano", "sigla_uf"], how="left")
        agg["gap_media_meta_uf"] = (agg["media_taxa_alfabetizacao"] - agg["meta_uf_2030"]).round(2)

    return agg.sort_values(["sigla_uf", "ano"])


gold_evolucao_uf = build_gold_evolucao_uf(silver_mun)
print(f"Gold evolução UF: {len(gold_evolucao_uf):,} linhas")
display(gold_evolucao_uf.head(10))

### 3.3 Gold: Ranking de UFs

In [ ]:
def build_gold_ranking_uf(df: pd.DataFrame) -> pd.DataFrame:
    """Ranking de UFs pelo indicador no ano mais recente."""
    if not {"ano", "sigla_uf", "taxa_alfabetizacao"}.issubset(df.columns):
        return pd.DataFrame()

    ano_max = df["ano"].max()
    ranking = (
        df[df["ano"] == ano_max]
        .groupby("sigla_uf", dropna=False)
        .agg(
            ano                = ("ano",               "first"),
            media_taxa         = ("taxa_alfabetizacao", "mean"),
            total_municipios   = ("id_municipio",       "nunique"),
            municipios_em_risco= ("atingiu_meta_uf",
                                  lambda x: (~x.fillna(True)).sum()),
        )
        .reset_index()
        .round({"media_taxa": 2})
        .sort_values("media_taxa", ascending=False)
        .reset_index(drop=True)
    )
    ranking.index += 1
    ranking.index.name = "posicao"
    return ranking.reset_index()


gold_ranking_uf = build_gold_ranking_uf(silver_mun)
print(f"Gold ranking UF: {len(gold_ranking_uf)} UFs")
display(gold_ranking_uf)

### 3.4 Gold: Municípios em Risco (para políticas públicas)

In [ ]:
def build_gold_municipios_risco(df: pd.DataFrame, top_n: int = 100) -> pd.DataFrame:
    """
    Top N municípios com maior gap negativo em relação à meta UF 2030.
    Acionável para políticas públicas de intervenção prioritária.
    """
    if "gap_meta_uf_2030" not in df.columns:
        return pd.DataFrame()

    ano_max = df["ano"].max()
    risco = (
        df[(df["ano"] == ano_max) & df["gap_meta_uf_2030"].notna()]
        .groupby(["id_municipio", "sigla_uf"], dropna=False)
        .agg(
            taxa_alfabetizacao = ("taxa_alfabetizacao", "mean"),
            meta_uf_2030       = ("meta_uf_2030",       "first"),
            gap_meta_uf_2030   = ("gap_meta_uf_2030",   "mean"),
        )
        .reset_index()
        .round(2)
        .sort_values("gap_meta_uf_2030")
        .head(top_n)
    )
    risco["ano_referencia"] = ano_max
    return risco


gold_risco = build_gold_municipios_risco(silver_mun)
print(f"Gold municípios em risco: {len(gold_risco)} municípios (piores gaps)")
display(gold_risco.head(15))

### 3.5 Gold: Comparativo Nacional

In [ ]:
def build_gold_comparativo_nacional(meta_br: pd.DataFrame, ind_uf: pd.DataFrame) -> pd.DataFrame:
    """Evolução da taxa nacional vs trajetória de metas 2024-2030."""
    if meta_br.empty:
        return pd.DataFrame()

    df = meta_br.copy()
    if "ano" in df.columns:
        df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")

    # Derrete as colunas de meta em linhas para facilitar visualização
    meta_cols = [c for c in df.columns if c.startswith("meta_alfabetizacao_")]
    if meta_cols:
        df_melted = df.melt(
            id_vars=["ano", "rede", "taxa_alfabetizacao"],
            value_vars=meta_cols,
            var_name="meta_ano",
            value_name="meta_valor"
        )
        df_melted["meta_ano"] = df_melted["meta_ano"].str.extract(r"(\d{4})").astype("Int64")
        df_melted["taxa_alfabetizacao"] = pd.to_numeric(df_melted["taxa_alfabetizacao"], errors="coerce")
        df_melted["meta_valor"] = pd.to_numeric(df_melted["meta_valor"], errors="coerce")
        return df_melted

    return df


gold_nacional = build_gold_comparativo_nacional(meta_brasil, silver_uf)
print(f"Gold comparativo nacional: {len(gold_nacional):,} linhas")
display(gold_nacional.dropna(subset=["meta_valor"]).head(10))

## 4. Persistência da Camada Gold

In [ ]:
gold_datasets = {
    "indicador_municipio" : gold_municipio,
    "evolucao_temporal_uf": gold_evolucao_uf,
    "ranking_uf"          : gold_ranking_uf,
    "municipios_risco"    : gold_risco,
    "comparativo_nacional": gold_nacional,
}


def save_gold(name, df):
    if df.empty:
        print(f"  Gold '{name}' vazio — pulando.")
        return

    # Salva local
    out = LOCAL_GOLD / f"{name}.parquet"
    df.to_parquet(out, index=False, engine="pyarrow")
    print(f"  Local: {out} ({len(df):,} linhas)")

    # Upload S3
    if USE_AWS:
        s3_client = boto3.client("s3", region_name=AWS_REGION)
        buffer = io.BytesIO()
        df.to_parquet(buffer, index=False, engine="pyarrow")
        buffer.seek(0)
        key = f"gold/{name}/{name}.parquet"
        s3_client.put_object(Bucket=S3_BUCKET, Key=key, Body=buffer.getvalue())
        print(f"  S3:   s3://{S3_BUCKET}/{key}")


print("Salvando camada Gold:")
for name, df in gold_datasets.items():
    save_gold(name, df)

print(f"\nGold concluído. Modo: {'AWS S3' if USE_AWS else 'LOCAL'}")

## 5. Análise Exploratória Gold

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["figure.dpi"] = 120

# Top 10 e Bottom 10 UFs
if not gold_ranking_uf.empty and "media_taxa" in gold_ranking_uf.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    top10 = gold_ranking_uf.nlargest(10, "media_taxa")
    bot10 = gold_ranking_uf.nsmallest(10, "media_taxa")

    axes[0].barh(top10["sigla_uf"], top10["media_taxa"], color="#2ecc71")
    axes[0].set_title("Top 10 UFs — Maior Taxa de Alfabetização")
    axes[0].set_xlabel("Taxa de Alfabetização (%)")
    axes[0].invert_yaxis()

    axes[1].barh(bot10["sigla_uf"], bot10["media_taxa"], color="#e74c3c")
    axes[1].set_title("Bottom 10 UFs — Menor Taxa de Alfabetização")
    axes[1].set_xlabel("Taxa de Alfabetização (%)")
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.savefig(LOCAL_GOLD / "ranking_uf.png", bbox_inches="tight")
    plt.show()
    print("Gráfico salvo em gold/ranking_uf.png")

In [ ]:
# Evolução nacional vs meta
if not gold_nacional.empty and "taxa_alfabetizacao" in gold_nacional.columns:
    fig, ax = plt.subplots(figsize=(10, 5))

    # Taxa realizada por ano
    taxa_real = (
        gold_nacional.dropna(subset=["ano", "taxa_alfabetizacao"])
        .groupby("ano")["taxa_alfabetizacao"].mean()
    )
    ax.plot(taxa_real.index.astype(int), taxa_real.values,
            marker="o", linewidth=2, label="Taxa Realizada", color="#3498db")

    # Trajetória de meta
    meta_traj = (
        gold_nacional.dropna(subset=["meta_ano", "meta_valor"])
        .groupby("meta_ano")["meta_valor"].mean()
    )
    ax.plot(meta_traj.index.astype(int), meta_traj.values,
            marker="s", linestyle="--", linewidth=2,
            label="Trajetória Meta 2030", color="#e74c3c")

    ax.axhline(80, color="green", linestyle=":", alpha=0.7, label="Meta Final 2030 (80%)")
    ax.set_title("Evolução da Taxa de Alfabetização vs Meta Nacional")
    ax.set_xlabel("Ano")
    ax.set_ylabel("Taxa de Alfabetização (%)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(LOCAL_GOLD / "evolucao_nacional.png", bbox_inches="tight")
    plt.show()

## 6. Aplicação em Inteligência Artificial

A camada Gold está preparada para os seguintes casos de uso de IA:

In [ ]:
# Demonstração: feature matrix para modelo preditivo
print("Feature Matrix para Modelo Preditivo de Alfabetização")
print("-" * 55)

feature_cols = [
    "id_municipio", "sigla_uf", "ano",
    "taxa_alfabetizacao",      # TARGET
    "media_portugues",         # Feature: proficiência linguística
    "meta_mun_2030",           # Feature: nível de ambição da meta
    "gap_meta_uf_2030",        # Feature: gap atual
    "proporcao_aluno_nivel_0", # Feature: % alunos sem proficiência
    "proporcao_aluno_nivel_1",
]

feature_matrix = gold_municipio[[c for c in feature_cols if c in gold_municipio.columns]]
print(f"Shape da feature matrix: {feature_matrix.shape}")
print(f"\nVariável target: taxa_alfabetizacao")
print(f"Features disponíveis: {[c for c in feature_cols[3:] if c in feature_matrix.columns]}")
print()
print("Casos de uso:")
print("  1. Regressão: predizer taxa_alfabetizacao do próximo ano")
print("  2. Classificação: identificar municípios que atingirão a meta 2030")
print("  3. Clustering: agrupar municípios por perfil de risco educacional")
print("  4. Série temporal: projetar trajetória para cada UF")

display(feature_matrix.dropna().head(5))

---
## Resumo Gold Layer

| Dataset | Linhas | Uso Principal |
|---|---|---|
| `indicador_municipio` | ~23k | Dashboards, análise por município |
| `evolucao_temporal_uf` | ~54 | Séries históricas, tendências |
| `ranking_uf` | 27 | Comparativo entre estados |
| `municipios_risco` | 100 | Priorização de políticas públicas |
| `comparativo_nacional` | ~20 | Monitoramento da meta 2030 |

**Próximo passo:** execute `04_streaming_simulation.ipynb`